<a href="https://colab.research.google.com/github/RCDS13/PI-JP-4SEM/blob/main/etl_transito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
!ls "/content/drive/MyDrive/PI_JP"

!pip install rpy2
%load_ext rpy2.ipython

Mounted at /content/drive
dataset_final.csv     datatran2022.csv	datatran2025.csv
dataset_final_SP.csv  datatran2023.csv	dicionario_variaveis.csv
datatran2021.csv      datatran2024.csv	dicionario_variaveis.docx


In [2]:
%%R

library(dplyr)
library(stringi)
library(tidyverse)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.1.6
✔ ggplot2   4.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.4     ✔ tibble    3.3.1
✔ purrr     1.2.1     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [5]:
%%R

path <- "/content/drive/MyDrive/PI_JP/"

# PADRONIZACAO E LEITURA
df1 <- read_delim(paste0(path, "datatran2021.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df2 <- read_delim(paste0(path, "datatran2022.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df3 <- read_delim(paste0(path, "datatran2023.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df4 <- read_delim(paste0(path, "datatran2024.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df5 <- read_delim(paste0(path, "datatran2025.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))

# JUNCAO DE TABELAS
df_final <- bind_rows(df1, df2, df3, df4, df5)

# SALVAR TABELA FINAL
write_excel_csv(df_final, paste0(path, "dataset_final.csv"), delim = ";")

# DIMENSAO DA TABELA FINAL
dim(df_final)

Rows: 64567 Columns: 30
── Column specification ────────────────────────────────────────────────────────
Delimiter: ";"
chr  (17): dia_semana, uf, km, municipio, causa_acidente, tipo_acidente, cla...
dbl  (10): id, br, pessoas, mortos, feridos_leves, feridos_graves, ilesos, i...
num   (1): longitude
date  (1): data_inversa
time  (1): horario

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 64606 Columns: 30
── Column specification ────────────────────────────────────────────────────────
Delimiter: ";"
chr  (17): dia_semana, uf, km, municipio, causa_acidente, tipo_acidente, cla...
dbl  (10): id, br, pessoas, mortos, feridos_leves, feridos_graves, ilesos, i...
num   (1): longitude
date  (1): data_inversa
time  (1): horario

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Ro

In [6]:
%%R
# VERIFICA LINHAS DUPLICADAS

total_original <- nrow(df_final)

total_unicas <- nrow(distinct(df_final))

duplicados <- total_original - total_unicas

print(paste("Registros duplicados encontrados:", duplicados))

[1] "Registros duplicados encontrados: 0"


In [7]:
%%R

# CARREGA TABELA FINAL

dados <- read.csv2("/content/drive/MyDrive/PI_JP/dataset_final.csv", fileEncoding="UTF-8")

# REMOCAO DE ACENTOS
dados_sp <- dados %>%
  filter(uf == "SP") %>%
  mutate(across(where(is.character), ~ stri_trans_general(., "Latin-ASCII")))

# SALVAMENTO USANDO FILE ENCONDING LATIN1 PRA COMTABILIDADE
write.csv2(dados_sp,
           "/content/drive/MyDrive/PI_JP/dataset_final_SP.csv",
           row.names = FALSE,
           fileEncoding = "latin1")

In [ ]:
%%R

# ESCOPO DO DICINARIO DE VARIAVEIS
dicionario <- data.frame(
  Variavel = names(dados),
  Tipo_R = sapply(dados, class),
  Descricao = "",
  Valores = "",
  stringsAsFactors = FALSE
)

# VISUALIZACAO
print(dicionario)

# SALVA NO DRIVE
write.csv2(dicionario,
           "/content/drive/MyDrive/PI_JP/dicionario_variaveis.csv",
           row.names = FALSE,
           fileEncoding = "UTF-8")

In [ ]:
%%R

print(dados_sp)

         id data_inversa    dia_semana  horario uf  br    km
1    331730   2021-01-01   sexta-feira 05:30:00 SP 116 453.0
2    332015   2021-01-02        sabado 10:20:00 SP 116 362.6
3    332229   2021-01-03       domingo 14:40:00 SP 153 266.0
4    332289   2021-01-03       domingo 18:10:00 SP 116 230.5
5    332397   2021-01-04 segunda-feira 07:25:00 SP 381  83.0
6    332695   2021-01-06  quarta-feira 06:40:00 SP 116 362.2
7    332889   2021-01-07  quinta-feira 09:50:00 SP 116 119.2
8    332955   2021-01-07  quinta-feira 16:10:00 SP 116 166.7
9    332969   2021-01-07  quinta-feira 12:20:00 SP 116 545.0
10   333016   2021-01-07  quinta-feira 21:50:00 SP 116 312.0
11   333088   2021-01-08   sexta-feira 09:10:00 SP 116 357.8
12   333091   2021-01-08   sexta-feira 11:45:00 SP 116 289.0
13   333107   2021-01-08   sexta-feira 12:20:00 SP 116 103.2
14   333151   2021-01-08   sexta-feira 16:50:00 SP 116 565.0
15   333397   2021-01-09        sabado 19:35:00 SP 116  38.5
16   333577   2021-01-10